# SOXX 2011–2018 历史披露目录来源

仅执行 `plan.json` 中四个 SEC 官方信托申报分片的只读请求，保存原字节、时间与 NAS 恢复。0 市场结果、0 拟合、0 账户。


In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex,time
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_legacy_basket_history_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['parent_raw_index_sha256']==sha(root/'data/snapshots/stage1_soxx_legacy_filing_index_20261023/submissions_CIK0001100663.json')
rawdir=root/'data/snapshots/stage1_soxx_legacy_basket_history_20261023';rawdir.mkdir(parents=True,exist_ok=True)
claim=card/'attempt_started.json';receipt_path=card/'transport_result.json'
assert not claim.exists() and not receipt_path.exists() and not any(rawdir.iterdir())
claim.write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'maximum_GETs':4},indent=2)+'\n')
results=[];candidate_rows=[]
for j,url in enumerate(plan['urls']):
 if j:time.sleep(plan['request']['minimum_seconds_between_requests'])
 name=url.rsplit('/',1)[-1];raw=rawdir/name;headers=rawdir/(name+'.headers.txt')
 started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['request']['connect_timeout_seconds']),
  '--max-time',str(plan['request']['total_timeout_seconds_each']),'--max-filesize',str(plan['request']['max_response_bytes_each']),
  '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
  '-H','User-Agent: '+plan['request']['user_agent'],'-H','Accept: application/json',url]
 proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['request']['total_timeout_seconds_each']+15)
 ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
 item={'url':url,'requested_at_utc':started,'received_at_utc':ended,'http_status':status,
  'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:400],
  'raw_path':str(raw.relative_to(root)),'raw_sha256':sha(raw) if raw.exists() else None,
  'raw_bytes':raw.stat().st_size if raw.exists() else 0,'headers_sha256':sha(headers) if headers.exists() else None,
  'headers_bytes':headers.stat().st_size if headers.exists() else 0}
 results.append(item)
 if proc.returncode or status!=200 or item['final_url']!=url or not raw.exists() or raw.stat().st_size>plan['request']['max_response_bytes_each']:
  item['stop_reason']='transport_or_identity_failed';break
 try:
  data=json.loads(raw.read_bytes());required=['accessionNumber','filingDate','reportDate','acceptanceDateTime','form','primaryDocument']
  lengths={key:len(data[key]) for key in required};assert len(set(lengths.values()))==1 and all(isinstance(data[key],list) for key in required)
  dates=data['filingDate'];item['rows']=len(dates);item['actual_range']=[min(dates),max(dates)];item['advertised_range']=plan['index_advertised_ranges'][j]
  item['range_metadata_equal']=item['actual_range']==item['advertised_range']
  forms=set(plan['relevant_forms'])
  matches=[]
  for i in range(len(dates)):
   if data['form'][i] in forms and dates[i][:4] in {str(y) for y in plan['target_filing_years']}:
    matches.append({key:data[key][i] for key in required})
  item['candidate_form_rows']=len(matches);candidate_rows.extend(matches)
 except Exception as exc:
  item['stop_reason']='schema_invalid_'+type(exc).__name__+': '+str(exc)[:100];break
receipt={'status':'four_official_trust_sidecars_received_metadata_only' if len(results)==4 and all('stop_reason' not in x for x in results) else 'source_batch_stopped',
 'plan_sha256':sha(card/'plan.json'),'GETs_consumed':len(results),'requests':results,'candidate_trust_rows':len(candidate_rows),
 'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'SOXX_historical_baskets_qualified':False}
receipt_path.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='soxx-legacy-index-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in [claim,receipt_path,*sorted(rawdir.iterdir())]:
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='soxx-legacy-index-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   for item in results:
    if item['raw_sha256'] is None:continue
    member=stored.extractfile(Path(item['raw_path']).name)
    assert member and hashlib.sha256(member.read()).hexdigest()==item['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=True
 receipt_path.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
(card/'candidate_trust_filings.json').write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'rows':candidate_rows,'SOXX_series_qualified':False},ensure_ascii=False,indent=2)+'\n')
print({'status':receipt['status'],'GETs':len(results),'trust_candidates':len(candidate_rows),'NAS_restore':True})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'four_official_trust_sidecars_received_metadata_only', 'GETs': 4, 'trust_candidates': 188, 'NAS_restore': True}


## 中文结论

执行后见 `summary.json` 和 `conclusion.md`。信托申报目录不能证明 SOXX 系列持仓。


In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex,time
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_legacy_basket_history_20261023';plan=json.loads((card/'2011_01_31_body_identity_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['candidate_index_sha256']==sha(card/'candidate_trust_filings.json')
rawdir=root/'data/snapshots/stage1_soxx_legacy_basket_history_20261023/2011_01_31_bodies';rawdir.mkdir(parents=True,exist_ok=True)
claim=card/'2011_01_31_attempt_started.json';receipt_path=card/'2011_01_31_transport_result.json'
assert not claim.exists() and not receipt_path.exists() and not any(rawdir.iterdir())
claim.write_text(json.dumps({'plan_sha256':sha(card/'2011_01_31_body_identity_plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'max_GETs':2},indent=2)+'\n')
requests=[]
for j,item in enumerate(plan['candidates']):
 if j:time.sleep(plan['request']['minimum_seconds_between_requests'])
 url=item['url'];name=item['accession']+'_'+item['primary_document'];raw=rawdir/name;headers=rawdir/(name+'.headers.txt')
 started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['request']['connect_timeout_seconds']),
  '--max-time',str(plan['request']['total_timeout_seconds_each']),'--max-filesize',str(plan['request']['max_response_bytes_each']),
  '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
  '-H','User-Agent: '+plan['request']['user_agent'],'-H','Accept: text/html',url]
 proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['request']['total_timeout_seconds_each']+15)
 ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
 entry={'accession':item['accession'],'url':url,'started_at_utc':started,'finished_at_utc':ended,'http_status':status,
  'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:400],
  'raw_path':str(raw.relative_to(root)),'raw_sha256':sha(raw) if raw.exists() else None,'raw_bytes':raw.stat().st_size if raw.exists() else 0,
  'headers_sha256':sha(headers) if headers.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0}
 requests.append(entry)
 if proc.returncode or status!=200 or entry['final_url']!=url or not raw.exists() or raw.stat().st_size>plan['request']['max_response_bytes_each']:
  entry['stop_reason']='transport_or_identity_failed';break
receipt={'status':'both_exact_2011_NQ_bodies_received_identity_pending' if len(requests)==2 and all('stop_reason' not in x for x in requests) else 'paired_body_source_stopped',
 'plan_sha256':sha(card/'2011_01_31_body_identity_plan.json'),'GETs_consumed':len(requests),'requests':requests,
 'SOXX_body_identity_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
receipt_path.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='soxx-2011-nq-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in [claim,receipt_path,*sorted(rawdir.iterdir())]:
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='soxx-2011-nq-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   for entry in requests:
    if entry['raw_sha256'] is None:continue
    member=stored.extractfile(Path(entry['raw_path']).name)
    assert member and hashlib.sha256(member.read()).hexdigest()==entry['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=True
 receipt_path.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
print({'status':receipt['status'],'GETs':len(requests),'bytes':[x['raw_bytes'] for x in requests],'NAS_restore':True})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'both_exact_2011_NQ_bodies_received_identity_pending', 'GETs': 2, 'bytes': [64106, 8974684], 'NAS_restore': True}


In [1]:
import json,hashlib,re,html
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_legacy_basket_history_20261023';plan=json.loads((card/'2011_01_31_body_identity_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
transport=json.loads((card/'2011_01_31_transport_result.json').read_text())
assert transport['plan_sha256']==sha(card/'2011_01_31_body_identity_plan.json') and transport['nas_scoped_restore_verified']
assert len(transport['requests'])==2 and not (card/'2011_01_31_identity_result.json').exists()
checks=[]
for item in transport['requests']:
 raw=root/item['raw_path'];payload=raw.read_bytes();assert hashlib.sha256(payload).hexdigest()==item['raw_sha256']
 lower=payload.lower()
 counts={word:lower.count(word.encode()) for word in ('soxx','phlx','sox','semiconductor')}
 title_hits=[]
 for title in ('iShares PHLX SOX Semiconductor Sector Index Fund','PHLX SOX Semiconductor Sector Index Fund'):
  pos=lower.find(title.lower().encode())
  if pos>=0:
   fragment=payload[pos:pos+2500].decode('latin1',errors='replace')
   clean=' '.join(html.unescape(re.sub(r'<[^>]*>',' ',fragment)).split())
   title_hits.append({'title':title,'byte_offset':pos,'report_date_near_title':'January 31, 2011' in clean[:800],
    'security_shares_value_header_near_title':all(x in clean[:1800] for x in ('Security','Shares','Value'))})
 qualified=any(x['report_date_near_title'] and x['security_shares_value_header_near_title'] for x in title_hits)
 checks.append({'accession':item['accession'],'raw_sha256':item['raw_sha256'],'raw_bytes':item['raw_bytes'],
  'target_term_counts':counts,'target_title_hits':title_hits,'fixed_identity_qualified':qualified})
result={'status':'both_2011_01_31_trust_NQ_bodies_fail_fixed_SOXX_title_identity' if not any(x['fixed_identity_qualified'] for x in checks) else 'one_or_more_2011_01_31_SOXX_NQ_schedule_titles_found',
 'plan_sha256':sha(card/'2011_01_31_body_identity_plan.json'),'checks':checks,
 'source_NAS_archive_sha256':transport['nas_evidence']['sha256'],'source_NAS_scoped_restore_verified':True,
 'SOXX_schedule_selected_accession':next((x['accession'] for x in checks if x['fixed_identity_qualified']),None),
 'GETs_consumed':2,'new_GETs_for_identity':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'interpretation':'Neither fixed N-Q body can be treated as SOXX based on semiconductor company or sector rows alone. This does not prove no other trust filing for the January 2011 report date contains SOXX.'}
(card/'2011_01_31_identity_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'selected':result['SOXX_schedule_selected_accession']})


{'status': 'both_2011_01_31_trust_NQ_bodies_fail_fixed_SOXX_title_identity', 'selected': None}
